# 🎓 Student Performance Predictor
### A Beginner-Friendly Machine Learning Project

---

## What are we building?
We will build a model that **predicts a student's final exam score** based on:
- 📚 How many hours they study per day
- 🏫 Their attendance percentage
- 📝 Their previous exam scores

## How does Machine Learning work here?
1. We show the computer many **examples** (past student data)
2. The computer **learns patterns** from those examples
3. When we give it a **new student's info**, it predicts their score!

---

## Step 1: Import Libraries

Libraries are like **toolboxes** — each one gives us special tools to work with.
- `pandas` → works with tables (like Excel)
- `matplotlib` / `seaborn` → draws graphs
- `sklearn` → machine learning tools

In [ ]:
# --- Import all the tools we need ---

import pandas as pd                  # For loading and working with our data table
import numpy as np                   # For math operations
import matplotlib.pyplot as plt      # For drawing basic graphs
import seaborn as sns                # For prettier graphs

# Machine Learning tools from scikit-learn
from sklearn.model_selection import train_test_split   # To split data into train & test
from sklearn.tree import DecisionTreeRegressor         # Our ML model
from sklearn.metrics import mean_absolute_error        # To measure how good our model is
from sklearn import tree                               # To visualize the decision tree

print("✅ All libraries imported successfully!")

---
## Step 2: Load the Dataset

We load our student data from a CSV file (a simple spreadsheet).
The data has **50 students** with their study habits and scores.

In [ ]:
# --- Load the data from our CSV file ---
# pd.read_csv() reads a CSV file and turns it into a table called a "DataFrame"

df = pd.read_csv('data/student_data.csv')

print("✅ Data loaded successfully!")
print(f"\n📊 Dataset has {df.shape[0]} rows (students) and {df.shape[1]} columns (features)")
print("\n--- First 5 rows of our data ---")
df.head()  # Show the first 5 rows

---
## Step 3: Explore the Data

Before training, let's understand what our data looks like.

In [ ]:
# --- Get basic statistics about our data ---
# This shows min, max, average, etc. for each column

print("📈 Basic Statistics of our Dataset:")
print("=" * 50)
df.describe().round(2)

In [ ]:
# --- Check the data types and if anything is missing ---
print("🔍 Data Types and Non-Null Counts:")
print("=" * 40)
print(df.info())

---
## Step 4: Data Cleaning

Real-world data often has **missing values** or **mistakes**.
We need to clean it before training our model.

In [ ]:
# --- Check for missing values ---
# Missing values are like empty cells in Excel - our model can't handle them

print("🧹 Checking for Missing Values:")
print("=" * 35)
missing = df.isnull().sum()
print(missing)

if missing.sum() == 0:
    print("\n✅ Great! No missing values found. Our data is clean!")
else:
    print("\n⚠️  Found missing values. Filling with column averages...")
    df.fillna(df.mean(), inplace=True)  # Replace missing values with the average
    print("✅ Missing values filled!")

In [ ]:
# --- Check for duplicate rows ---
# Duplicate rows mean the same student appears twice — that could confuse our model

duplicates = df.duplicated().sum()
print(f"🔁 Duplicate rows found: {duplicates}")

if duplicates > 0:
    df.drop_duplicates(inplace=True)  # Remove duplicates
    print("✅ Duplicates removed!")
else:
    print("✅ No duplicates found!")

print(f"\n📊 Final dataset size: {df.shape[0]} students")

---
## Step 5: Visualize the Data 📊

Graphs help us **see patterns** in the data.
For example: Do students who study more really score higher?

In [ ]:
# --- Graph 1: Distribution of Final Scores ---
# This shows how scores are spread across all students

plt.figure(figsize=(8, 5))
plt.hist(df['final_score'], bins=10, color='steelblue', edgecolor='white', linewidth=0.8)
plt.title('Distribution of Final Scores', fontsize=15, fontweight='bold')
plt.xlabel('Final Score', fontsize=12)
plt.ylabel('Number of Students', fontsize=12)
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

print(f"📌 Average final score: {df['final_score'].mean():.1f}")
print(f"📌 Lowest score: {df['final_score'].min()}")
print(f"📌 Highest score: {df['final_score'].max()}")

In [ ]:
# --- Graph 2: Study Hours vs Final Score ---
# Scatter plot: each dot = one student
# We want to see if more study hours = higher score

plt.figure(figsize=(8, 5))
plt.scatter(df['study_hours'], df['final_score'],
            color='coral', edgecolors='white', linewidth=0.5, s=80, alpha=0.85)
plt.title('Study Hours vs Final Score', fontsize=15, fontweight='bold')
plt.xlabel('Daily Study Hours', fontsize=12)
plt.ylabel('Final Score', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate correlation: 1.0 = perfect relationship, 0 = no relationship
corr = df['study_hours'].corr(df['final_score'])
print(f"📌 Correlation between study hours and final score: {corr:.2f}")
print("   (1.0 = perfect match, 0 = no relation)")

In [ ]:
# --- Graph 3: Attendance vs Final Score ---

plt.figure(figsize=(8, 5))
plt.scatter(df['attendance'], df['final_score'],
            color='mediumseagreen', edgecolors='white', linewidth=0.5, s=80, alpha=0.85)
plt.title('Attendance % vs Final Score', fontsize=15, fontweight='bold')
plt.xlabel('Attendance (%)', fontsize=12)
plt.ylabel('Final Score', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

corr2 = df['attendance'].corr(df['final_score'])
print(f"📌 Correlation between attendance and final score: {corr2:.2f}")

In [ ]:
# --- Graph 4: Correlation Heatmap ---
# This shows how strongly each feature is related to the others
# Darker = stronger relationship

plt.figure(figsize=(7, 5))
numeric_df = df[['study_hours', 'attendance', 'previous_scores', 'final_score']]
sns.heatmap(numeric_df.corr().round(2),
            annot=True,          # Show numbers on the map
            cmap='YlOrRd',       # Color scheme
            linewidths=0.5,
            fmt='.2f')
plt.title('Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("📌 Values close to 1.0 mean strong positive relationship")
print("   Values close to 0 mean weak or no relationship")

---
## Step 6: Prepare Data for Machine Learning

We need to:
1. **Separate** inputs (features) from output (what we want to predict)
2. **Split** data into training set and testing set

Think of it like:
- 🏋️ **Training set** = practice problems the model learns from
- 📝 **Test set** = surprise exam to check if it really learned!

In [ ]:
# --- Separate features (inputs) and target (output) ---

# X = the input columns our model will use to make predictions
X = df[['study_hours', 'attendance', 'previous_scores']]

# y = the answer column (what we want the model to predict)
y = df['final_score']

print("🎯 Input features (X):")
print(X.head(3))
print("\n🏆 Target output (y):")
print(y.head(3).to_string())

In [ ]:
# --- Split into Training and Testing sets ---
# test_size=0.2 means 20% goes to testing, 80% goes to training
# random_state=42 makes the split the same every time we run (reproducibility)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% for testing
    random_state=42     # So results are the same every run
)

print("✅ Data split complete!")
print(f"\n📚 Training set size: {X_train.shape[0]} students (used to TEACH the model)")
print(f"📝 Testing set size:  {X_test.shape[0]} students (used to TEST the model)")
print(f"\n   That's {X_train.shape[0]/len(X)*100:.0f}% for training and {X_test.shape[0]/len(X)*100:.0f}% for testing")

---
## Step 7: Train the Machine Learning Model 🤖

We use a **Decision Tree** — it works like a game of 20 questions!

Example logic the tree might learn:
```
If study_hours > 5:
    If attendance > 85:
        Predict high score (90+)
    Else:
        Predict medium score (75)
Else:
    Predict lower score (50)
```

`max_depth=4` limits how many questions the tree can ask (prevents over-fitting).

In [ ]:
# --- Create and train the Decision Tree model ---

# Create the model
# max_depth=4 means the tree can only ask 4 questions deep
# This prevents the model from "memorizing" the training data
model = DecisionTreeRegressor(max_depth=4, random_state=42)

# Train the model: show it the training data and let it learn
# .fit() is the magic function that does the learning!
model.fit(X_train, y_train)

print("🤖 Model training complete!")
print("\n📖 What just happened?")
print("   The Decision Tree looked at 40 students' data")
print("   and learned patterns like:")
print("   'Students who study more tend to score higher'")
print("   'High attendance often means better final scores'")

In [ ]:
# --- Visualize the Decision Tree ---
# This shows the actual questions the tree asks to make predictions

plt.figure(figsize=(20, 8))
tree.plot_tree(
    model,
    feature_names=['study_hours', 'attendance', 'previous_scores'],
    filled=True,          # Color the boxes
    rounded=True,         # Rounded corners
    fontsize=9
)
plt.title('Decision Tree — How the Model Makes Decisions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("🌳 Each box = one question the model asks")
print("   Deeper boxes = more specific conditions")
print("   The final leaf boxes = the predicted score")

---
## Step 8: Evaluate the Model 📏

Now we check: **how good is our model?**

We use the **test set** (data the model has NEVER seen) to see if its predictions are close to the real scores.

In [ ]:
# --- Make predictions on the test set ---
# .predict() uses what the model learned to guess scores

y_pred = model.predict(X_test)

# --- Compare predicted vs actual scores ---
results = pd.DataFrame({
    'Actual Score':    y_test.values,
    'Predicted Score': y_pred.round(1),
    'Difference':      abs(y_test.values - y_pred).round(1)
})

print("🔍 Predicted vs Actual Scores (Test Set):")
print("=" * 45)
print(results.to_string(index=False))

In [ ]:
# --- Calculate Mean Absolute Error (MAE) ---
# MAE = average difference between predicted and actual scores
# Example: MAE of 3 means our predictions are off by about 3 points on average

mae = mean_absolute_error(y_test, y_pred)

# R² Score: measures what % of the variation the model explains
# 1.0 = perfect, 0.0 = model learned nothing
r2 = model.score(X_test, y_test)

print("📊 Model Performance Results:")
print("=" * 40)
print(f"\n📌 Mean Absolute Error (MAE): {mae:.2f} points")
print(f"   → On average, predictions are off by only {mae:.1f} points")
print(f"\n📌 R² Score: {r2:.4f}")
print(f"   → The model explains {r2*100:.1f}% of score variation")

print("\n💡 In Simple Words:")
print(f"   If a student's real score is 80,")
print(f"   our model would guess somewhere around {80-mae:.0f} to {80+mae:.0f}.")
print(f"   That's pretty good for a beginner ML model! 🎉")

In [ ]:
# --- Visualize: Actual vs Predicted ---
# If our model were perfect, all dots would fall on the diagonal line

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, color='steelblue', edgecolors='white',
            linewidth=0.5, s=100, alpha=0.85, zorder=3)

# Draw the "perfect prediction" line
min_val = min(y_test.min(), y_pred.min()) - 2
max_val = max(y_test.max(), y_pred.max()) + 2
plt.plot([min_val, max_val], [min_val, max_val],
         color='tomato', linestyle='--', linewidth=2, label='Perfect Prediction')

plt.title('Actual vs Predicted Scores', fontsize=15, fontweight='bold')
plt.xlabel('Actual Score', fontsize=12)
plt.ylabel('Predicted Score', fontsize=12)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("📌 Dots close to the red line = accurate predictions")
print("   Dots far from the line = less accurate predictions")

In [ ]:
# --- Feature Importance: Which input matters most? ---
# This tells us which feature (study hours, attendance, or previous score)
# had the biggest effect on predictions

importance = model.feature_importances_
features = ['Study Hours', 'Attendance', 'Previous Scores']
colors = ['steelblue', 'mediumseagreen', 'coral']

plt.figure(figsize=(7, 4))
bars = plt.bar(features, importance, color=colors, edgecolor='white', linewidth=0.8)
plt.title('Feature Importance — What Matters Most?', fontsize=14, fontweight='bold')
plt.ylabel('Importance Score', fontsize=11)
plt.ylim(0, 1)

# Add value labels on bars
for bar, val in zip(bars, importance):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.01,
             f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

most_important = features[importance.argmax()]
print(f"\n🏆 Most important feature: {most_important}")
print("   This feature had the biggest impact on predictions!")

---
## Step 9: Predict for a New Student 🎓

Now let's use our trained model to predict scores for **new students** (data the model has never seen)!

In [ ]:
# --- Predict for custom student inputs ---
# Change these values to predict for different students!

def predict_score(study_hours, attendance, previous_score):
    """
    This function takes a student's info and predicts their final score.
    
    Parameters:
    - study_hours: How many hours per day the student studies (e.g., 3.5)
    - attendance: Attendance percentage (e.g., 85)
    - previous_score: Score in the last exam (e.g., 72)
    """
    # Create a small table with the student's data
    student_data = pd.DataFrame({
        'study_hours':     [study_hours],
        'attendance':      [attendance],
        'previous_scores': [previous_score]
    })
    
    # Use the model to predict
    predicted = model.predict(student_data)[0]
    
    return round(predicted, 1)


# --- Try different student profiles ---

print("🎓 Student Score Predictions")
print("=" * 50)

# Student 1: Hard worker
score1 = predict_score(study_hours=6.0, attendance=92, previous_score=85)
print(f"\n👩‍🎓 Student A (Hardworking):")
print(f"   Study: 6 hrs/day | Attendance: 92% | Prev Score: 85")
print(f"   🔮 Predicted Final Score: {score1}")

# Student 2: Average student
score2 = predict_score(study_hours=3.5, attendance=78, previous_score=68)
print(f"\n👨‍🎓 Student B (Average):")
print(f"   Study: 3.5 hrs/day | Attendance: 78% | Prev Score: 68")
print(f"   🔮 Predicted Final Score: {score2}")

# Student 3: Struggling student
score3 = predict_score(study_hours=1.0, attendance=50, previous_score=42)
print(f"\n😰 Student C (Struggling):")
print(f"   Study: 1 hr/day | Attendance: 50% | Prev Score: 42")
print(f"   🔮 Predicted Final Score: {score3}")

In [ ]:
# --- Interactive: Enter YOUR values ---
# Run this cell and enter your own student data!

print("📝 Predict YOUR Score!")
print("=" * 35)

try:
    my_study   = float(input("Enter study hours per day (e.g. 4.5): "))
    my_attend  = float(input("Enter attendance % (e.g. 85): "))
    my_prev    = float(input("Enter previous score (e.g. 70): "))
    
    my_score = predict_score(my_study, my_attend, my_prev)
    
    print(f"\n🎯 Predicted Final Score: {my_score} / 100")
    
    if my_score >= 85:
        print("🌟 Excellent! Keep up the great work!")
    elif my_score >= 70:
        print("✅ Good job! A bit more effort can take you to the top!")
    elif my_score >= 55:
        print("📚 Not bad! Try to study more and attend classes regularly.")
    else:
        print("💪 There's room to improve! Focus on attendance and study time.")
        
except ValueError:
    print("⚠️  Please enter valid numbers!")

---
## 🎉 Summary: What We Built

Congratulations! Here's what you accomplished:

| Step | What We Did |
|------|-------------|
| 1 | Imported libraries (pandas, matplotlib, sklearn) |
| 2 | Loaded student data from CSV |
| 3 | Explored data with statistics |
| 4 | Cleaned data (handled missing values, duplicates) |
| 5 | Visualized data with graphs |
| 6 | Split data into training (80%) and testing (20%) |
| 7 | Trained a Decision Tree model |
| 8 | Evaluated using MAE and R² score |
| 9 | Predicted scores for new students |

## 🧠 Key Concepts Learned
- **Features (X)**: The inputs used to make predictions
- **Target (y)**: The value we want to predict
- **Training set**: Data the model learns from
- **Test set**: New data to evaluate the model
- **MAE**: Average prediction error in the same units as your target
- **Decision Tree**: A model that asks yes/no questions to predict values

## 🚀 Next Steps
- Try changing `max_depth` (e.g., 2 or 6) and see how MAE changes
- Add more features like `sleep_hours` or `assignments_completed`
- Try other models like `RandomForestRegressor` or `LinearRegression`
- Collect real student data and retrain!